In [9]:
import pandas as pd 

from sklearn.metrics import make_scorer, brier_score_loss
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectKBest, f_classif

from skopt import BayesSearchCV
from xgboost import XGBClassifier

from itertools import product

import warnings
warnings.simplefilter(action = "ignore", category = RuntimeWarning)

input_file_path = '../input'
output_file_path = '../output'

# Build and Tune Final Models + Predictions

In [3]:
def feature_select_stats(season_data, tourney_data, features, model):
    """
    This method uses a chi-squared test to evaluate the top features of a given model. 
    It iterates from 5 to the number of supplied features to identify the optimal number of features to 
    minimize the brier score, the test metric for the competition.

    :param season_data: Regular season data used to train the model.
    :param tourney_data: Tournament data used to test the model.
    :param features: List of features to evaluate.
    :param model: Model to select features for.
    :return: Game-level data with stats weighted by day of the season (later is higher weight).
    """
   
    X_train = season_data[features]
    y_train = season_data['Pred']
    
    best_brier_score = 1
    best_feature_set = []
    
    for n in range(5, len(features)): 
        print(f"Model with top {n} features...")
        # Select top k features based on the chi-squared test
        selector = SelectKBest(f_classif, k=n)  
        selector.fit(X_train, y_train)
        statistical_feats = list(X_train.columns[selector.get_support()])
    
        # Train and evaluate model with selected features
        X_train_stat = season_data[statistical_feats].fillna(0)
        y_train_stat = season_data['Pred']
        
        X_test_stat = tourney_data[statistical_feats].fillna(0)
        y_test_stat = tourney_data['Pred']
    
        model.fit(X_train_stat, y_train_stat)
    
        y_pred_proba = model.predict_proba(X_test_stat)[:, 1]
        brier_score = brier_score_loss(y_test_stat, y_pred_proba)
        # Identifies if we've hit a new low brier score with the given feature set.
        if brier_score < best_brier_score:
            best_brier_score = brier_score
            best_feature_set = statistical_feats
            print("NEW BEST")
            
        print(f"Brier Score: {brier_score:.4f}")
    
    print("Best Features: ", best_feature_set)

    return best_feature_set

In [ ]:
def hyper_parameter_tuning(season_data, features, model, param_grid):
    """
    This method use BayesSearchCV to tune the Hyperparameters of an XGBoost model
    to minimize the competiton evaluation metric (brier score).

    :param season_data: Regular season data used to train the model.
    :param features: List of features to evaluate.
    :param model: Model to optimize.
    ::param param_grid: Dictionary of paramaters to tune.
    """
    brier_scorer = make_scorer(brier_score_loss, greater_is_better=False)

    X_train = season_data[features]
    y_train = season_data['Pred']
    
    bayes_search = BayesSearchCV(
        estimator=model,
        search_spaces=param_grid,
        n_iter=50,  # Number of iterations
        cv=3,  # 3-fold cross-validation
        scoring=brier_scorer,  # Custom scorer
        n_jobs=-1,
        random_state=42
    )
    
    # Fit the search
    bayes_search.fit(X_train, y_train)
    
    best_params = bayes_search.best_params_ # Get the best parameters
    print("Best Parameters:", bayes_search.best_params_)
    print("Best Brier Score:", -bayes_search.best_score_)

In [16]:
def evaluate_model(prob, test):
    """
    This method evaluates the effects of rounding win probabilties at different threshols to 1 or 0
    to see if it helps further minimize the actual brier score from test data.

    :param prob: List of raw predicted probabilities.
    :param test: Actual game outcomes to calculate brier score against.
    """
    adj_prob = [0, .05, .1, .15, .2, .25, .3, .35, .4, .45, .5]
    for adj in adj_prob:
        prob_adj = [round(x) if x <= adj or 1 - adj <= x else x 
                            for x in prob]
        brier_score = brier_score_loss(test, prob_adj) 
        print(f"Brier score (Adj={adj}):", brier_score)

In [5]:
def final_predictions(teams, stats, season, features, model, correction=0):
    """
    This method creates final competition compliant outpout with formated IDs and final win probabilities.

    :param teams: Dataframe of all teams to calculate win probabilties for.
    :param stats: Feature data with stats used for modeling.
    :param season: Season to predict win probabilties for.
    :param features: Features used for the model.
    :param model: Model to use for predicting probabilities.
    :param correction: Threshold to round probabilities to 1 or 0. 
    :return: Final output with competition compliant ID and win probability of the first team in the ID.
    """
    # Create all possibile matchs for teams in the data set. 
    team_combos =  pd.DataFrame(product(teams['TeamID'], teams['TeamID']), columns=['team1', 'team2'])
    team_combos = team_combos[team_combos["team1"]!=team_combos["team2"]]
    
    # Team with lower TeamID is first in the final ID
    team_combos["TeamID_first"] = team_combos[['team1', 'team2']].min(axis=1)
    team_combos["TeamID_second"] = team_combos[['team1', 'team2']].max(axis=1)
    team_combos["Season"] = season

    # Create final ID, drop duplicates since each match up will appear twice.
    team_combos['ID'] = team_combos['Season'].astype('str') + '_' + team_combos['TeamID_first'].astype('str') + '_' + team_combos['TeamID_second'].astype('str') 
    team_combos = team_combos[["TeamID_first", "TeamID_second", "Season", "ID"]].drop_duplicates()
    
    # Join Stats data onto match ups, find the difference between Team1 stats and Team 2 Stats.
    team_stats = team_combos.merge(
        stats[stats["Season"]==season],
        left_on=['Season', 'TeamID_first'],
        right_on=['Season', 'TeamId'],
        how='inner'
    ).merge(
        stats[stats["Season"]==season],
        left_on=['Season', 'TeamID_second'],
        right_on=['Season', 'TeamId'],
        how='inner',
        suffixes=('_first', '_second')
    )
    
    for feat in features:
        team_stats[feat] = team_stats[feat+'_first'] - team_stats[feat+'_second']
    
    # Use supplied Model to predict Win Probabilities. Round based on corrections.
    predictions = model.predict_proba(team_stats[features])[:, 1]
    predictions = [round(x) if x <= correction or 1 - correction <= x else x for x in predictions]
    
    # Format final output
    team_stats["Pred"] = predictions
    final_output = team_stats[["ID", "Pred"]]

    return final_output


## Men's Predictions

### Baseline

In [6]:
# For the mens model, I found XGBoost gave the best results (compared to simple LogisticRegression and RandomForest), 
# So that is the base model I decided to buld from.

tourney_data = pd.read_csv(f'{output_file_path}/TournamentDataModel.csv')
rs_data = pd.read_csv(f'{output_file_path}/RegularDataModel.csv')

tourney_data_m = tourney_data[tourney_data['League'] == 'M'].fillna(0)
rs_data_m = rs_data[rs_data['League'] == 'M'].fillna(0)

features =  ['Score', 'Score_against', 'FGper', 'FG3per', 'FTper', 'FGper_against', 'FG3per_against', 'FTper_against',
           "OEFF", "DEFF", "NET_EFF", "eFG", "TS", "ORper", "DRper", "TOper", "AST_TO", "3P_Reliance", "FTR", "STLper", "avg_rank"]

X_train = rs_data_m[features].fillna(0)
y_train = rs_data_m['Pred']

X_test = tourney_data_m[features].fillna(0)
y_test = tourney_data_m['Pred']

model =  XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=6)
model.fit(X_train, y_train)

y_pred_proba = model.predict_proba(X_test)[:, 1]
brier_score = brier_score_loss(y_test, y_pred_proba)
print(f"Brier Score: {brier_score:.4f}")

Brier Score: 0.1923


### Feature Selection + Hyper-parameter Tuning

In [7]:
best_features = feature_select_stats(rs_data_m, tourney_data_m, features, model)

Model with top 5 features...
NEW BEST
Brier Score: 0.1931
Model with top 6 features...
NEW BEST
Brier Score: 0.1927
Model with top 7 features...
Brier Score: 0.1929
Model with top 8 features...
NEW BEST
Brier Score: 0.1926
Model with top 9 features...
Brier Score: 0.1927
Model with top 10 features...
NEW BEST
Brier Score: 0.1922
Model with top 11 features...
Brier Score: 0.1923
Model with top 12 features...
NEW BEST
Brier Score: 0.1920
Model with top 13 features...
Brier Score: 0.1925
Model with top 14 features...
Brier Score: 0.1923
Model with top 15 features...
Brier Score: 0.1921
Model with top 16 features...
Brier Score: 0.1922
Model with top 17 features...
Brier Score: 0.1924
Model with top 18 features...
Brier Score: 0.1922
Model with top 19 features...
Brier Score: 0.1924
Model with top 20 features...
Brier Score: 0.1924
Best Features:  ['Score', 'Score_against', 'FGper', 'FG3per', 'FGper_against', 'FG3per_against', 'OEFF', 'NET_EFF', 'eFG', 'TS', 'AST_TO', 'avg_rank']


In [ ]:
# Define the parameter space for BayesSearchCV
param_grid = {
    "n_estimators": (50, 500),
    "learning_rate": (0.01, 0.3, "log-uniform"),
    "max_depth": (3, 12),
    "subsample": (0.5, 1.0),
    "colsample_bytree": (0.5, 1.0),
    "reg_lambda": (1e-3, 10, "log-uniform"),  
    "reg_alpha": (1e-3, 10, "log-uniform"),  
    "gamma": (0, 1.0),                       
    "min_child_weight": (1, 10)             
}


# Set up the XGBoost mode'
xgb = XGBClassifier(objective="binary:logistic", eval_metric="logloss", use_label_encoder=False)

hyper_parameter_tuning(rs_data_m, tourney_data_m, best_features, xgb, param_grid)

### Buld Final Model

In [9]:
# Train the final model using the best parameters
X_train = rs_data_m[best_features].fillna(0)
y_train = rs_data_m['Pred']

X_test = tourney_data_m[best_features].fillna(0)
y_test = tourney_data_m['Pred']

final_model = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    learning_rate=0.1,
    max_depth=5,
    n_estimators=500,
    reg_lambda=0.10023, 
    gamma=1,              # Minimum loss reduction for split
    min_child_weight=1,
    random_state=42
)

# Train the final model
final_model.fit(X_train, y_train)

# Evaluate the model (for example, using Brier Score Loss)
y_pred = final_model.predict_proba(X_test)[:, 1]  # Probabilities for class 1
brier_score = brier_score_loss(y_test, y_pred)

print(f'Brier Score: {brier_score}')
evaluate_model(y_pred, y_test)

Brier Score: 0.1917843673706415
Brier score (Adj=0): 0.1917843673706415
Brier score (Adj=0.05): 0.19176140614047377
Brier score (Adj=0.1): 0.19201013731468544
Brier score (Adj=0.15): 0.19187071782063578
Brier score (Adj=0.2): 0.19296978548103544
Brier score (Adj=0.25): 0.19372821591477582
Brier score (Adj=0.3): 0.19591505450019364
Brier score (Adj=0.35): 0.2064053153469445
Brier score (Adj=0.4): 0.22355686666106953
Brier score (Adj=0.45): 0.2597479227871795
Brier score (Adj=0.5): 0.29160636758321273


### Make Predictions

In [11]:
# Make Final Output
teams_m = pd.read_csv(f'{input_file_path}/MTeams.csv')
combined_data = pd.read_csv(f'{output_file_path}/CombinedSeasonStats.csv')
teams_m = teams_m[teams_m["LastD1Season"]==2025]

final_predictions_m = final_predictions(teams_m, combined_data, 2025, best_features, final_model, .05) 
final_predictions_m.to_csv(f'{output_file_path}/final_predictions_m.csv')

## Women's Predictions

### Baseline

In [11]:
# For the womens model, I found Logistic Regression performed better than XGBoost at baseline, 
# so I decided to tune that for final predoctions, which resulted in a slightly different process
# than the mens model. 

tourney_data = pd.read_csv(f'{output_file_path}/TournamentDataModel.csv')
rs_data = pd.read_csv(f'{output_file_path}/RegularDataModel.csv')

tourney_data_w = tourney_data[tourney_data['League'] == 'W'].fillna(0)
rs_data_w = rs_data[rs_data['League'] == 'W'].fillna(0)

features =  ['Score', 'Score_against', 'FGper', 'FG3per', 'FTper', 'FGper_against', 'FG3per_against', 'FTper_against',
           "OEFF", "DEFF", "NET_EFF", "eFG", "TS", "ORper", "DRper", "TOper", "AST_TO", "3P_Reliance", "FTR", "STLper"]

X_train = rs_data_w[features].fillna(0)
y_train = rs_data_w['Pred']

X_test = tourney_data_w[features].fillna(0)
y_test = tourney_data_w['Pred']

model =  LogisticRegression()
model.fit(X_train, y_train)

y_pred_proba = model.predict_proba(X_test)[:, 1]
brier_score = brier_score_loss(y_test, y_pred_proba)
print(f"Brier Score: {brier_score:.4f}")

Brier Score: 0.2091


/Users/michael/Documents/Data Projects/ncaa_predictions/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


### Feature Selection + Hyper-parameter Tuning

In [12]:
best_features = feature_select_stats(rs_data_w, tourney_data_w, features, model)

Model with top 5 features...
NEW BEST
Brier Score: 0.2127
Model with top 6 features...
NEW BEST
Brier Score: 0.2119
Model with top 7 features...
Brier Score: 0.2142
Model with top 8 features...
Brier Score: 0.2140
Model with top 9 features...
Brier Score: 0.2136
Model with top 10 features...
Brier Score: 0.2136
Model with top 11 features...
Brier Score: 0.2135
Model with top 12 features...
Brier Score: 0.2137
Model with top 13 features...
Brier Score: 0.2122
Model with top 14 features...
Brier Score: 0.2122
Model with top 15 features...
NEW BEST
Brier Score: 0.2114
Model with top 16 features...
Brier Score: 0.2117
Model with top 17 features...
NEW BEST
Brier Score: 0.2105
Model with top 18 features...


/Users/michael/Documents/Data Projects/ncaa_predictions/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


NEW BEST
Brier Score: 0.2089
Model with top 19 features...
Brier Score: 0.2093
Best Features:  ['Score', 'Score_against', 'FGper', 'FG3per', 'FTper', 'FGper_against', 'FG3per_against', 'OEFF', 'DEFF', 'NET_EFF', 'eFG', 'TS', 'ORper', 'DRper', 'TOper', 'AST_TO', 'FTR', 'STLper']


/Users/michael/Documents/Data Projects/ncaa_predictions/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [ ]:
# Define the parameter space for BayesSearchCV
param_grid = {
    'penalty': ['l2'],  # L1 for Lasso, L2 for Ridge
    'C': (0.001, 1000, 'log-uniform'),  # Inverse of regularization strength
    'fit_intercept': [True, False],
    'max_iter': (50, 500),
}

# Set up the logistic regression model
log_reg = LogisticRegression() 

hyper_parameter_tuning(rs_data_w, tourney_data_w, best_features, log_reg, param_grid)

### Buld Final Model

In [ ]:
# Train the final model using the best parameters
X_train = rs_data_w[best_features]
y_train = rs_data_w['Pred']

X_test = tourney_data_w[best_features]
y_test = tourney_data_w['Pred']

final_model = LogisticRegression(
    penalty='l2', C=300.0, fit_intercept=False, max_iter=1000
)

# Train the final model
final_model.fit(X_train, y_train)

# Evaluate the model (for example, using Brier Score Loss)
y_pred = final_model.predict_proba(X_test)[:, 1]  # Probabilities for class 1
brier_score = brier_score_loss(y_test, y_pred)

print(f'Brier Score: {brier_score}')
evaluate_model(y_pred, y_test)

Brier Score: 0.20801148911587275
Brier score (Adj=0): 0.20801148911587275
Brier score (Adj=0.05): 0.20809904867761678
Brier score (Adj=0.1): 0.20860107467453967
Brier score (Adj=0.15): 0.2094315437207119
Brier score (Adj=0.2): 0.21396498807476994
Brier score (Adj=0.25): 0.21905924700329107
Brier score (Adj=0.3): 0.22668996572646605
Brier score (Adj=0.35): 0.23537155502679658
Brier score (Adj=0.4): 0.25201037467973303
Brier score (Adj=0.45): 0.2690054388937015
Brier score (Adj=0.5): 0.3782771535580524


### Make Predictions

In [17]:
teams_w = pd.read_csv(f'{input_file_path}/WTeams.csv')
combined_data = pd.read_csv(f'{output_file_path}/CombinedSeasonStats.csv')

final_predictions_w = final_predictions(teams_w, combined_data.fillna(0), 2025, best_features, final_model, 0)
final_predictions_w.to_csv(f'{output_file_path}/final_predictions_w.csv')
final_predictions_w

,ID,Pred
0,2025_3101_3102,0.518409
1,2025_3101_3103,0.802374
2,2025_3101_3104,0.132750
3,2025_3101_3105,0.521068
4,2025_3101_3106,0.969089
...,...,...
65336,2025_3477_3479,0.249008
65337,2025_3477_3480,0.332528
65338,2025_3478_3479,0.198963
65339,2025_3478_3480,0.271772


# Create Final Submission File

In [3]:
final_predictions_m = pd.read_csv(f'{output_file_path}/final_predictions_m.csv')
final_predictions_w = pd.read_csv(f'{output_file_path}/final_predictions_w.csv')

final_ouput = pd.concat([final_predictions_m, final_predictions_w])[['ID', 'Pred']]
final_ouput.to_csv(f'{output_file_path}/submission.csv')
final_ouput

,ID,Pred
0,2025_1101_1102,0.767824
1,2025_1101_1103,0.084825
2,2025_1101_1104,0.000000
3,2025_1101_1105,0.848485
4,2025_1101_1106,0.609445
...,...,...
65336,2025_3477_3479,0.249008
65337,2025_3477_3480,0.332528
65338,2025_3478_3479,0.198963
65339,2025_3478_3480,0.271772
